# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

# Show main high-level fields in the metadata
print("\nHigh-level metadata fields:")
keys_to_show = [
    "name", "identifier", "version", "description", "keywords", "dataCollection",
    "dataBiases", "dataLimitations", "dataCollectionTimeframe", "dataPreprocessingProtocol" 
]
for k in keys_to_show:
    value = getattr(metadata, k, None)
    print(f"- {k}: {value}")

## 2. Data Overview
Review available record sets and list their fields, columns, and their `@id`. This helps identify data available for extraction.

In [ ]:
# List all RecordSets using their @id
print("Available RecordSets and their IDs:")
recordsets = dataset.metadata.record_sets
for rs in recordsets:
    print(f"- RecordSet '@id': {rs.id}, name: {rs.name}")

# Show associated fields and their @id for each record set
for rs in recordsets:
    print(f"\nRecordSet '@id': {rs.id} -> fields:")
    for field in rs.fields:
        colinfo = field.column.id if hasattr(field, 'column') and field.column else None
        dtype = field.data_type if hasattr(field, 'data_type') else ""
        print(f"    - Field '@id': {field.id}, name: {field.name}, dataType: {dtype}, column: {colinfo}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Choose the primary RecordSet for participant survey responses
# Replace with the appropriate RecordSet '@id' obtained above.
# For demonstration, we'll take the first available RecordSet.
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]
print(f"RecordSet IDs found: {record_sets_ids}")
primary_record_set_id = record_sets_ids[0]

# Extract data from all record sets
dataframes = {}
for record_set_id in record_sets_ids:
    print(f"Extracting records from RecordSet '@id': {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns in {record_set_id}:", df.columns.tolist())
    print(df.head(2))

# Work with the DataFrame for the primary record set
df = dataframes[primary_record_set_id]
print(f"\nFirst few rows of '@id' {primary_record_set_id} DataFrame:")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's select the GAD-7 total score field as a numeric variable to demonstrate simple transformations. You can similarly explore PHQ-9, PSQ, and other fields.

In [ ]:
# Identify the GAD-7 total score field using its '@id'
# Adjust the exact field name if needed based on column listing, e.g., 'gad7_total', or its '@id'.
numeric_field = None
for col in df.columns:
    if 'gad' in col.lower() and 'total' in col.lower():
        numeric_field = col
        break
if not numeric_field:
    numeric_field = df.select_dtypes(include='number').columns[0]  # fallback
print(f"Using numeric field: {numeric_field}")

# Set a threshold for demonstration
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with '{numeric_field}' > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() else 1)
print(f"Normalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical attribute if available
# E.g., 'gender' or 'level_of_education' -- adjust field name as per actual column
possible_group_fields = [c for c in df.columns if c.lower() in ['gender', 'sex', 'level_of_education']]
group_field = possible_group_fields[0] if possible_group_fields else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped mean '{numeric_field}' by '{group_field}':")
    print(grouped_df.head())
else:
    print("No obvious categorical group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field (e.g. GAD-7 total score)
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field].dropna(), bins=10, kde=True)
plt.title(f"Distribution of '{numeric_field}'")
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# If categorical grouping exists, show grouped boxplot
if group_field:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x=group_field, y=numeric_field)
    plt.title(f"'{numeric_field}' by '{group_field}'")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
This notebook demonstrated end-to-end loading, exploration, and simple analysis of the FAIR² Mental Health Survey Dataset from Kilifi County using the `mlcroissant` library. Key variables such as GAD-7 (anxiety) were analyzed, and the Croissant schema enabled programmatic discovery of data structure.

**Key Observations:**
- Metadata and data structure can be easily inspected via entity `@id`s.
- Exploratory analysis is enabled for fields such as mental health assessment scores.
- For publication-quality analysis, carefully validate field selection using the printed `@id` and adjust column selection logic as needed.
- This approach can be extended to other Croissant datasets with minimal adjustments.
